# 시작하기: 관리형 VPC Lattice를 사용하는 Private API Gateway

이 실습에서는 모의 통합이 구성된 프라이빗 [Amazon API Gateway](https://docs.aws.amazon.com/apigateway/latest/developerguide/apigateway-private-apis.html)를 배포한 다음, 관리형 VPC 송신을 사용하여 [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)에 연결합니다.

[API-VPCE DNS 형식](https://docs.aws.amazon.com/apigateway/latest/developerguide/apigateway-private-api-create.html)(`{api-id}-{vpce-id}.execute-api.{region}.amazonaws.com`)은 퍼블릭 DNS에서 확인할 수 있으며 유효한 AWS 관리형 TLS 인증서를 사용합니다.

VPC 송신과 관리형 VPC 리소스에 관한 배경 정보는 [프로젝트 README](../README.md)와 [관리형 VPC 리소스](./README.md)를 참조하세요.

## 아키텍처

![아키텍처](./images/api-gw.png)

## 사전 요구 사항

- [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb) 완료(VPC 및 AgentCore Gateway 배포)

이 실습에는 도메인 이름이나 ACM 인증서가 필요하지 않습니다.

## 1단계: 종속성 설치 및 라이브러리 가져오기

In [ ]:
import os
from pathlib import Path

# 프로젝트 루트로 이동
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

!pip install --force-reinstall -q -r requirements.txt

In [ ]:
import json
import os
import time

import boto3
from utils.utils import get_token

# 실습 0에서 변수 복원
%store -r ACCOUNT_A_ID
%store -r ACCOUNT_A_PROFILE
%store -r GATEWAY_ID
%store -r GATEWAY_URL
%store -r USER_POOL_ID
%store -r USER_POOL_CLIENT_ID
%store -r TOKEN_ENDPOINT_URL
%store -r OAUTH_SCOPES
%store -r VPC_USW2_ID
%store -r VPC_USW2_PRIVATE_SUBNETS

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID

REGION = "us-west-2"
session = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION)
agentcore = session.client("bedrock-agentcore-control")

# Cognito 클라이언트 보안 암호 가져오기
cognito = session.client("cognito-idp")
client_desc = cognito.describe_user_pool_client(UserPoolId=USER_POOL_ID, ClientId=USER_POOL_CLIENT_ID)
CLIENT_SECRET = client_desc["UserPoolClient"]["ClientSecret"]

print(f"Account:    {ACCOUNT_A_ID}")
print(f"Region:     {REGION}")
print(f"Gateway ID: {GATEWAY_ID}")
print(f"VPC ID:     {VPC_USW2_ID}")

## 2단계: Private API Gateway 배포

이 CDK 스택은 다음 리소스를 배포합니다.
- 모의 통합(`/health` GET, `/items` GET/POST)이 구성된 **Private API Gateway**
- 프라이빗 DNS가 활성화된 프라이빗 서브넷의 `execute-api`용 **VPC 엔드포인트**
- VPC CIDR에서 인바운드 HTTPS(443)를 허용하는 **보안 그룹**

VPC 엔드포인트를 프라이빗 API와 연결하면 API Gateway에서 특수한 DNS 이름을 생성합니다.
```
https://{api-id}-{vpce-id}.execute-api.{region}.amazonaws.com/{stage}
```

이 DNS 이름은 **퍼블릭 DNS에서 확인 가능**하며(VPCE 프라이빗 IP로 확인됨), 유효한 AWS 관리형 TLS 인증서를 사용합니다.

In [ ]:
!cdk deploy PrivateApigw --profile {ACCOUNT_A_PROFILE} --require-approval never --outputs-file apigw-outputs.json

In [ ]:
with open("apigw-outputs.json") as f:
    apigw_outputs = json.load(f)["PrivateApigw"]

API_ID = apigw_outputs["ApiId"]
API_KEY_ID = apigw_outputs["ApiKeyId"]
VPCE_ID = apigw_outputs["VpceId"]
VPCE_SG_ID = apigw_outputs["VpceSgId"]

# API-VPCE DNS 형식 사용: 퍼블릭 DNS에서 확인 가능하며 VPCE 프라이빗 IP로 확인됨
API_VPCE_DNS = f"{API_ID}-{VPCE_ID}.execute-api.{REGION}.amazonaws.com"

# API 키 값 가져오기
apigw_client = session.client("apigateway")
api_key_response = apigw_client.get_api_key(apiKey=API_KEY_ID, includeValue=True)
API_KEY_VALUE = api_key_response["value"]

print(f"API ID:       {API_ID}")
print(f"API Key ID:   {API_KEY_ID}")
print(f"VPCE ID:      {VPCE_ID}")
print(f"API-VPCE DNS: {API_VPCE_DNS}")
print(f"VPCE SG:      {VPCE_SG_ID}")

## 3단계: API 키 자격 증명 공급자 생성

AgentCore Gateway가 API Gateway에 인증하려면 API 키가 필요합니다. API 키를 저장하고 AgentCore가 어떤 헤더로 키를 전송할지 지정하는 [API 키 자격 증명 공급자](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-identity.html)를 생성합니다.

API Gateway는 `x-api-key` 헤더에서 키를 받습니다.

In [ ]:
# AgentCore에 API 키 자격 증명 공급자 생성
cred_response = agentcore.create_api_key_credential_provider(
    name="private-apigw-api-key",
    apiKey=API_KEY_VALUE,
)
CRED_PROVIDER_ARN = cred_response["credentialProviderArn"]
print(f"Credential provider ARN: {CRED_PROVIDER_ARN}")

## 4단계: AgentCore Gateway 대상 생성

[관리형 VPC 리소스](./README.md)를 사용하여 Gateway 대상을 생성합니다. 엔드포인트는 퍼블릭 DNS에서 확인할 수 있는 API-VPCE DNS 형식을 사용합니다.

`credentialProviderConfigurations` 파라미터는 API Gateway에 인증할 때 API 키 자격 증명 공급자를 사용하도록 AgentCore에 지정합니다.

> **보안 그룹:** Resource Gateway ENI가 포트 443을 통해 VPCE에 연결할 수 있도록 VPCE 보안 그룹을 `securityGroupIds`에 전달합니다. [보안 그룹 고려 사항](./README.md#security-group-considerations)을 참조하세요.

In [ ]:
# OpenAPI 스키마를 로드하고 서버 URL 삽입
with open("01-managed-vpc-resource/openapi-private-apigw.json") as f:
    openapi_schema = json.load(f)

# 서버 URL을 실제 API-VPCE DNS 엔드포인트로 설정
TARGET_ENDPOINT = f"https://{API_VPCE_DNS}/prod"
openapi_schema["servers"] = [{"url": TARGET_ENDPOINT}]

OPENAPI_SCHEMA = json.dumps(openapi_schema)
print(f"Loaded OpenAPI schema: {openapi_schema['info']['title']} v{openapi_schema['info']['version']}")
print(f"Server URL: {TARGET_ENDPOINT}")
print(f"Endpoints: {list(openapi_schema['paths'].keys())}")

print(f"\nTarget endpoint: {TARGET_ENDPOINT}")

response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="private-apigw",
    description="Private API Gateway via VPCE and managed VPC egress",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "inlinePayload": OPENAPI_SCHEMA,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": CRED_PROVIDER_ARN,
                    "credentialParameterName": "x-api-key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
    privateEndpoint={
        "managedVpcResource": {
            "vpcIdentifier": VPC_USW2_ID,
            "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
            "endpointIpAddressType": "IPV4",
            "securityGroupIds": [VPCE_SG_ID],
        }
    },
)

TARGET_ID = response["targetId"]
print(f"\nTarget ID: {TARGET_ID}")
print(f"Status:    {response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nTarget is active!")
        print(f"  Managed resources: {target.get('privateEndpointManagedResources', {})}")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

## 5단계: AgentCore Gateway를 통해 API 호출

Cognito에서 액세스 토큰을 가져온 다음, Gateway를 통해 Private API Gateway의 작업을 MCP 도구로 호출합니다.

In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]
print(f"Access token obtained (expires in {token_response['expires_in']}s)")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

# 사용 가능한 도구 목록 확인(API 작업이 MCP 도구로 노출됨)
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
print("Available tools:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# 상태 확인
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "private-apigw___healthCheck", "arguments": {}},
        "id": 2,
    },
)
print("Health check:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# 항목 목록 확인
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "private-apigw___listItems", "arguments": {}},
        "id": 3,
    },
)
print("Items:")
print(json.dumps(response.json(), indent=2))

## 정리

1. Gateway 대상 삭제
2. API 키 자격 증명 공급자 삭제
3. API Gateway CDK 스택 제거
4. VPCE 보안 그룹 삭제(VPC Lattice ENI가 계속 참조할 수 있으므로 CDK에서 유지됨)

> **참고:** AgentCore의 관리형 Resource Gateway ENI가 계속 참조할 수 있으므로 스택을 삭제할 때 VPCE 보안 그룹은 유지됩니다. Gateway 대상이 완전히 제거된 후 수동으로 삭제하세요.

In [ ]:
# # 1단계: Gateway target 삭제
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# print(f"Deleting target: {TARGET_ID}")
# while True:
#     try:
#         t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
#         print(f"  Status: {t['status']}")
#         time.sleep(15)
#     except agentcore.exceptions.ResourceNotFoundException:
#         print("  Target deleted.")
#         break

# # 2단계: credential provider 삭제
# agentcore.delete_api_key_credential_provider(name="private-apigw-api-key")
# print("Deleted credential provider: private-apigw-api-key")

In [ ]:
# # 3단계: stack 제거(SG는 유지되며 다음 cell에서 삭제)
# !cdk destroy PrivateApigw --profile {ACCOUNT_A_PROFILE} --force

In [ ]:
# # 4단계: 유지된 VPCE security group 삭제
# # "DependencyViolation"으로 실패하면 ENI가 해제될 때까지 몇 분 기다리세요.
# ec2_client = session.client("ec2")
# try:
#     ec2_client.delete_security_group(GroupId=VPCE_SG_ID)
#     print(f"Deleted security group: {VPCE_SG_ID}")
# except ec2_client.exceptions.ClientError as e:
#     if "DependencyViolation" in str(e):
#         print(f"SG {VPCE_SG_ID} still has dependencies. Wait a few minutes and retry.")
#     else:
#         raise